# 🚀 malayaLLM v2.3.0: 3B Malayalam Fine-Tuning on Google Colab
### Turnkey QLoRA Fine-Tuning with Unsloth (Free T4 GPU / 16GB VRAM)

This notebook fine-tunes **Qwen 2.5 3B Instruct** (or **Llama 3.2 3B Instruct**) on our curated Malayalam dataset (combining Alpaca SFT, CulturaX IndicCorp, Malayalam Wikipedia, and custom harvested records).

**Prerequisites:**
1. Ensure GPU is active: `Runtime` -> `Change runtime type` -> **T4 GPU**.
2. Upload `train.jsonl` to the Colab files sidebar (left panel folder icon).

In [ ]:
# @title 1. Install Unsloth & Core Optimization Libraries
!pip install --quiet --upgrade unsloth xformers trl peft accelerate datasets bitsandbytes

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Mount Google Drive (For Automatic Checkpoint Relay)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/malayaLLM_3B_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print('✓ Checkpoints will safely save to Google Drive:', CHECKPOINT_DIR)
except Exception as e:
    CHECKPOINT_DIR = './malayaLLM_3B_checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print('Drive skipped. Local directory:', CHECKPOINT_DIR)


In [ ]:
# @title 3. Load 4-Bit Base Model (Qwen 2.5 3B Instruct)
from unsloth import FastLanguageModel

# Qwen 2.5 has superior Dravidian tokenization (2x faster for Malayalam)
# Alternative 3B option: 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

print(f'Loading {MODEL_NAME} in 4-bit quantization...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None, # Auto-detects bfloat16 / float16
    load_in_4bit = True,
)
print('✓ Base model and tokenizer loaded successfully!')


In [ ]:
# @title 4. Attach QLoRA Adapters for Malayalam Morphology
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank (16 is ideal for 3B)
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 32,
    lora_dropout = 0, # 0 is optimized for Unsloth
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)
print('✓ QLoRA Adapters initialized on all projection layers!')


In [ ]:
# @title 5. Load and Prepare the Malayalam Training Dataset
from datasets import load_dataset
import os

# Check for uploaded train.jsonl
dataset_file = 'train.jsonl' if os.path.exists('train.jsonl') else 'malayalam_dataset_unified.jsonl'
if not os.path.exists(dataset_file):
    raise FileNotFoundError("Please upload 'train.jsonl' to the Colab files sidebar!")

raw_dataset = load_dataset('json', data_files=dataset_file, split='train')

chatml_template = """<|im_start|>system
നിങ്ങൾ ആധികാരികവും ശുദ്ധവുമായ മലയാളത്തിൽ സംസാരിക്കുന്ന ഒരു എഐ സഹായിയാണ്.<|im_end|>
<|im_start|>user
{instruction}{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

EOS_TOKEN = tokenizer.eos_token

def format_prompt(examples):
    instructions = examples.get('instruction', [])
    inputs = examples.get('input', [''] * len(instructions))
    outputs = examples.get('output', [])
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        inp_str = f"\nസന്ദർഭം: {inp}" if inp and str(inp).strip() else ""
        text = chatml_template.format(instruction=inst, input=inp_str, output=out) + EOS_TOKEN
        texts.append(text)
    return {'text': texts}

train_dataset = raw_dataset.map(format_prompt, batched=True)
print(f'✓ Successfully formatted {len(train_dataset)} Malayalam instruction pairs!')


In [ ]:
# @title 6. Launch 3B Fine-Tuning with Unsloth SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Effective batch = 8
        warmup_steps = 10,
        max_steps = 120, # ~2-3 epochs on dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'cosine',
        seed = 3407,
        output_dir = CHECKPOINT_DIR,
        save_strategy = 'steps',
        save_steps = 50,
    ),
)

print('🚀 Starting Fine-Tuning on T4 GPU...')
trainer_stats = trainer.train()
print('✓ Training complete!')


In [ ]:
# @title 7. Test Live Malayalam Generation
FastLanguageModel.for_inference(model)

prompt = chatml_template.format(
    instruction="കേരളത്തിലെ പരമ്പരാഗത ജലസംരക്ഷണ രീതികളെക്കുറിച്ചും കുളങ്ങളെക്കുറിച്ചും വിവരിക്കുക.",
    input="",
    output=""
)

inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, use_cache=True)

print('=== Response from Trained malayaLLM 3B ===')
print(tokenizer.batch_decode(outputs)[0])


In [ ]:
# @title 8. Save LoRA Adapters & Export GGUF for Local Ollama
import os

# 1. Save LoRA Adapter
lora_output_dir = os.path.join(CHECKPOINT_DIR, 'malayaLLM_3B_lora')
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print('✓ Saved LoRA Adapters to:', lora_output_dir)

# 2. Export GGUF (Q4_K_M) for 100% offline Ollama inference
gguf_output_dir = os.path.join(CHECKPOINT_DIR, 'malayaLLM_3B_GGUF')
try:
    model.save_pretrained_gguf(gguf_output_dir, tokenizer, quantization_method='q4_k_m')
    print('✓ GGUF Model successfully exported to:', gguf_output_dir)
    print('To run locally in Ollama: ollama create malayaLLM -f Modelfile')
except Exception as e:
    print(f'GGUF export notice: {e}')
